# Export โมเดลที่ดีที่สุดไปบอร์ด

1. สร้าง feature แบบเดียวกับ `ml_pipeline/03_window_history.ipynb` (ชุดที่คำนวณบนบอร์ดได้และไม่มี LDR leak)
2. เลือกโมเดลที่ export เป็น C array ได้ (DecisionTree, LogisticRegression, RandomForest) ด้วย leave-one-day-out F1 ต่างกัน ≤ 0.002 เลือกตัวที่ง่ายกว่า
3. save `model.joblib` + `RoomSensePredictor/model_weights.h`
4. คอมไพล์ `RoomSensePredictor.h` ด้วย g++ ป้อนแถวดิบ แล้วเช็กว่าผลตรงกับ Python ทุก window

In [1]:
import numpy as np, pandas as pd
train = pd.read_csv("../../data/train.csv")   # วันที่ 1–3
test = pd.read_csv("../../data/test.csv")     # วันที่ 4

In [2]:
def windows(df):
    """รวมทุก 20 แถว (10 s) ต่อ (day, boot) แล้วตัด warm-up 3 นาทีแรก (18 window)"""
    out = []
    for _, s in df.groupby(["day", "boot"], sort=False):
        s = s.assign(cam=s.person_count.where(s.person_count >= 0),                       # -1 = กล้องไม่ส่ง
                     door=((s.digital_do == 0) | (s.analog_2_raw > 1000)).astype(int),   # IR หรือ gray เห็นวัตถุที่ประตู
                     win=np.arange(len(s)) // 20)
        g = s.groupby("win")
        w = pd.DataFrame({
            "day": g.day.first(), "boot": g.boot.first(), "n": g.size(), "label": g.power_on.max(),
            "cam_max": g.cam.max(), "cam_mean": g.cam.mean(),
            "cam_pos_frac": g.cam.apply(lambda x: (x.dropna() > 0).mean() if x.notna().any() else np.nan),
            "cam_missing_frac": g.cam.apply(lambda x: x.isna().mean()),
            "door_any": g.door.max(),
            "door_events": g.door.apply(lambda x: int((x.diff() == 1).sum() + x.iloc[0])),
            "temp": g.temp_c.mean(), "hum": g.humidity.mean(), "mq2": g.mq2_raw.median(), "ldr": g.analog_1_raw.mean(),
        }).reset_index()
        # feature ย้อนหลัง: rolling มองแค่อดีต จึงคำนวณ real-time บนบอร์ดได้
        seen = w.cam_max.fillna(0)
        w["cam_max_roll6"] = seen.rolling(6, min_periods=1).max()               # 1 นาที
        w["cam_mean_roll3"] = w.cam_mean.rolling(3, min_periods=1).mean()       # 30 s
        w["door_recent_2m"] = w.door_any.rolling(12, min_periods=1).max()
        w["door_events_5m"] = w.door_events.rolling(30, min_periods=1).sum()
        w["temp_slope_5m"] = w.temp.diff(30) / 5
        i = pd.Series(np.arange(len(w)), dtype=float)
        w["since_cam_s"] = ((i - i.where(seen > 0).ffill()) * 10).clip(upper=900).fillna(900)
        w["since_door_s"] = ((i - i.where(w.door_any > 0).ffill()) * 10).clip(upper=900).fillna(900)
        out.append(w)
    W = pd.concat(out, ignore_index=True)
    return W[(W.n == 20) & ~((W.boot == 0) & (W.win < 18))].reset_index(drop=True)

tr, te = windows(train), windows(test)
F = ["cam_max", "cam_mean", "cam_pos_frac", "cam_missing_frac", "door_any", "door_events",
         "cam_max_roll6", "cam_mean_roll3", "door_recent_2m", "door_events_5m", "since_cam_s", "since_door_s"]
Xtr, ytr, days = tr[F], tr.label, tr.day
Xte, yte = te[F], te.label
Xtr.shape, Xte.shape

((3182, 12), (1061, 12))

In [3]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

cands = {   # เรียงจากง่ายไปซับซ้อน
    "DecisionTree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, class_weight="balanced", random_state=0),
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(50, max_depth=6, min_samples_leaf=5, class_weight="balanced", random_state=0, n_jobs=-1),
}
make = lambda clf: Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("clf", clf)])
cv = {n: cross_val_score(make(c), Xtr, ytr, groups=days, cv=LeaveOneGroupOut(), scoring="f1").mean() for n, c in cands.items()}
name = next(n for n in cands if cv[n] >= max(cv.values()) - 0.002)
model = make(cands[name]).fit(Xtr, ytr)
joblib.dump(model, "model.joblib")
print({k: round(v, 4) for k, v in cv.items()}, "->", name)

{'DecisionTree': np.float64(0.9796), 'LogisticRegression': np.float64(0.9734), 'RandomForest': np.float64(0.9861)} -> RandomForest


In [4]:
imp, sc, clf = model.named_steps.values()
arr = lambda typ, n, v: f"const {typ} {n}[] = {{{', '.join(repr(float(x)) if typ == 'double' else str(int(x)) for x in v)}}};\n"
h = (f"// generated by export_model.ipynb: {name} — อย่าแก้มือ\n#pragma once\n#include <stdint.h>\n"
     f"#define RS_WIN 20\n#define RS_N_FEATURES {len(F)}\n// features: {', '.join(F)}\n"
     + arr("double", "RS_MEDIAN", imp.statistics_) + arr("double", "RS_MEAN", sc.mean_) + arr("double", "RS_SCALE", sc.scale_))
if name == "LogisticRegression":
    h += "#define RS_LOGREG\n" + arr("double", "RS_COEF", clf.coef_[0]) + f"const double RS_INTERCEPT = {float(clf.intercept_[0])!r};\n"
else:
    feat, thr, left, right, p1, root = [], [], [], [], [], []
    for t in getattr(clf, "estimators_", [clf]):   # DecisionTree = ป่า 1 ต้น
        t, o = t.tree_, len(feat)
        root.append(o)
        feat += list(np.maximum(t.feature, 0)); thr += list(t.threshold)
        left += [c + o if c >= 0 else -1 for c in t.children_left]
        right += [c + o if c >= 0 else -1 for c in t.children_right]
        p1 += list(t.value[:, 0, 1] / t.value[:, 0].sum(1))
    h += (f"#define RS_N_TREES {len(root)}\n" + arr("int16_t", "RS_ROOT", root) + arr("int16_t", "RS_FEAT", feat)
          + arr("double", "RS_THR", thr) + arr("int16_t", "RS_LEFT", left) + arr("int16_t", "RS_RIGHT", right) + arr("double", "RS_P1", p1))
open("RoomSensePredictor/model_weights.h", "w").write(h)
print(f"{len(h) / 1024:.1f} KB")

62.1 KB


## ตรวจ: C++ บนเครื่อง vs Python
ป้อนแถวดิบทีละแถวเหมือนบอร์ด (reset object เมื่อ boot ใหม่) แล้วเทียบผลทุก window ทั้ง train และ test

In [5]:
import subprocess, tempfile, os
from sklearn.metrics import f1_score
cpp = r"""#include <cstdio>
#include "RoomSensePredictor.h"
int main() {
  RoomSensePredictor *p = nullptr; int day, boot, pc, ir, gray, key = -1;
  while (scanf("%d,%d,%d,%d,%d", &day, &boot, &pc, &ir, &gray) == 5) {
    if (day * 10 + boot != key) { delete p; p = new RoomSensePredictor(); key = day * 10 + boot; }
    long w = p->windows;
    p->update(pc, ir, gray);
    if (p->windows != w) printf("%d,%d,%ld,%d\n", day, boot, w, p->power);
  }
}"""
tmp = tempfile.mkdtemp()
open(f"{tmp}/h.cpp", "w").write(cpp)
subprocess.run(["g++", "-std=c++11", "-O2", "-I", os.path.abspath("RoomSensePredictor"), "h.cpp", "-o", "h"], cwd=tmp, check=True)

for split, raw, W in [("train", train, tr), ("test", test, te)]:
    out = subprocess.run([f"{tmp}/h"], input=raw[["day", "boot", "person_count", "digital_do", "analog_2_raw"]].to_csv(header=False, index=False),
                         capture_output=True, text=True, check=True).stdout
    c = pd.read_csv(pd.io.common.StringIO(out), names=["day", "boot", "win", "c_pred"])
    m = W.assign(py_pred=model.predict(W[F])).merge(c, on=["day", "boot", "win"])
    assert len(m) == len(W) and (m.py_pred == m.c_pred).all(), (split, len(m), len(W), (m.py_pred != m.c_pred).sum())
    print(f"{split}: C++ ตรงกับ Python {len(m)}/{len(W)} window, F1 = {f1_score(m.label, m.c_pred):.4f}")

train: C++ ตรงกับ Python 3182/3182 window, F1 = 0.9928
test: C++ ตรงกับ Python 1061/1061 window, F1 = 0.9748
